# CEREBRO PoC — 02F Human Review & Approval

**Experiment:** EXP-INGEST-005  
**Stage:** Artifact Intake  
**Purpose:** Human-in-the-Loop Validation

---

## Objective

Validate CEREBRO's human-in-the-loop control before an artifact is committed to the Digital Knowledge Twin.

AI-generated metadata is treated only as a **suggestion**.

The artifact owner remains the final authority and may:

- accept an AI suggestion,
- modify an AI suggestion,
- remove unsupported information,
- add missing information,
- approve or reject submission.

**Upload ≠ Ingest**

An artifact remains **STAGED** until human review is complete and explicit approval is provided.

### 1 — Create editable review copy

In [7]:
from pathlib import Path
import json
import copy

repo_root = Path.cwd().parents[1]

input_path = (
    repo_root
    / "poc/data/processed/EXP-INGEST-004-ai-prefill.json"
)

assert input_path.exists(), (
    f"02E output not found: {input_path}"
)

with open(
    input_path,
    "r",
    encoding="utf-8"
) as f:
    ai_prefill = json.load(f)

reviewed_prefill = copy.deepcopy(
    ai_prefill
)

print("✓ 02E AI enrichment loaded")
print("✓ Review copy created")
print("Fields:", list(reviewed_prefill.keys()))

✓ 02E AI enrichment loaded
✓ Review copy created
Fields: ['title', 'artifact_type', 'language', 'description', 'authors', 'people', 'organizations', 'projects', 'topics', 'tags']


### 2 — Human corrections

In [8]:
human_updates = {
    "title": "CEREBRO Digital Knowledge Twin",
    "artifact_type": "knowledge_description",
    "language": "English",
    "description": (
        "CEREBRO is a Digital Knowledge Twin designed to "
        "preserve and connect human knowledge while maintaining "
        "provenance to supporting source artifacts."
    ),
    "authors": [],
    "people": [],
    "organizations": [],
    "projects": ["CEREBRO"],
    "topics": [
        "Digital Knowledge Twin",
        "Provenance",
        "Knowledge Fragments",
        "Source Artifacts",
        "Assisted Recollection"
    ],
    "tags": [
        "CEREBRO",
        "knowledge",
        "provenance",
        "recollection"
    ]
}

### 3 — Apply human review while preserving AI provenance

In [ ]:
for field, human_value in human_updates.items():

    original_ai_value = reviewed_prefill[field]["value"]

    reviewed_prefill[field]["value"] = human_value
    reviewed_prefill[field]["user_confirmed"] = True # User has confirmed the value
    reviewed_prefill[field]["status"] = "confirmed"

    reviewed_prefill[field]["review"] = {
        "reviewer": "artifact_owner",
        "original_ai_value": original_ai_value,
        "final_value": human_value,
        "action": (
            "accepted"
            if original_ai_value == human_value
            else "modified"
        )
    }

print("✓ Human review applied")

✓ Human review applied


### 4 — Review before Submit

In [10]:
print("CEREBRO — Human Review")
print("=" * 60)

for field, metadata in reviewed_prefill.items():

    review = metadata["review"]

    print(f"\n{field}")
    print("  AI     :", review["original_ai_value"])
    print("  Final  :", review["final_value"])
    print("  Action :", review["action"])

print("\n" + "-" * 60)
print("User Review : COMPLETE")
print("Submit      : READY")

CEREBRO — Human Review

title
  AI     : CEREBRO
  Final  : CEREBRO Digital Knowledge Twin
  Action : modified

artifact_type
  AI     : Digital Knowledge Twin
  Final  : knowledge_description
  Action : modified

language
  AI     : 
  Final  : English
  Action : modified

description
  AI     : Preserves and connects human knowledge, maintaining provenance between fragments and their original source artifacts.
  Final  : CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge while maintaining provenance to supporting source artifacts.
  Action : modified

authors
  AI     : []
  Final  : []
  Action : accepted

people
  AI     : ['']
  Final  : []
  Action : modified

organizations
  AI     : [[''], '']
  Final  : []
  Action : modified

projects
  AI     : [[''], '']
  Final  : ['CEREBRO']
  Action : modified

topics
  AI     : [['Knowledge Management', 'Digital Twinning']]
  Final  : ['Digital Knowledge Twin', 'Provenance', 'Knowledge Fragments', 'Sour

### 5 — Approval gate

In [11]:
review_complete = all(
    metadata["user_confirmed"]
    for metadata in reviewed_prefill.values()
)

assert review_complete

user_approval = True

print("Review :", "COMPLETE")
print("Approval:", "APPROVED" if user_approval else "PENDING")
print("Submit :", "READY" if user_approval else "BLOCKED")

Review : COMPLETE
Approval: APPROVED
Submit : READY


### 6 - Persist user approval 

In [12]:
from pathlib import Path
import json

repo_root = Path.cwd().parents[1]

assert user_approval is True
assert "reviewed_prefill" in globals()

output_path = (
    repo_root
    / "poc/data/processed/EXP-INGEST-005-approved.json"
)

approved_review = {
    "experiment_id": "EXP-INGEST-005",
    "status": "approved",
    "user_approval": True,
    "fields": reviewed_prefill
}

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        approved_review,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ Human-approved artifact persisted")
print("Output:", output_path)

✓ Human-approved artifact persisted
Output: /Users/joeldizon/development/cerebro_dev/cerebro/poc/data/processed/EXP-INGEST-005-approved.json
